# iGov — Data Visualization & Predictive Analysis

> **Author:** Tomás Silva  
> **Dataset:** Dataset-iGov.csv  
> **Tools:** Python · Pandas · Matplotlib · Seaborn · Scikit-Learn · MLflow · ydata-profiling

---

## Project Overview

This notebook presents a **complete data analysis pipeline** applied to the iGov dataset, a public e-government services dataset tracking citizen interactions, response times, resolution rates, and satisfaction scores.

The goal is to **understand what drives citizen satisfaction** with digital government services and build predictive models to support operational decision-making.

---

## Table of Contents

1. [Imports & Setup](#1)
2. [Data Loading & Initial Exploration](#2)
3. [Data Preprocessing & Transformation](#3)
4. [Descriptive Statistics](#4)
5. [Integrated Statistical Dashboard](#5)
    - 5.1 [Correlation Heatmap](#5-1)
    - 5.2 [Histogram & KDE](#5-2)
    - 5.3 [Boxplot & Outliers](#5-3)
    - 5.4 [Bivariate Regression](#5-4)
    - 5.5 [K-Means Clustering](#5-5)
    - 5.6 [Decision Tree Classifier](#5-6)
    - 5.7 [Probability Distributions](#5-7)
6. [Scatterplot Matrix (Pairplot)](#6)
7. [Shape Analysis: Skewness & Kurtosis](#7)
8. [Automated EDA Report — ydata-profiling](#8)
9. [MLflow: Predictive Modelling](#9)
    - 9.1 [Model Training & Evaluation](#9-1)
    - 9.2 [Model Visualizations](#9-2)
10. [Decision Support System (SSD)](#10)
11. [Conclusions](#11)

---
## 1. Imports & Setup <a id="1"></a>

We begin by importing required libraries. Such as:

- **Pandas / NumPy** — data manipulation and numerical operations
- **Matplotlib / Seaborn** — static visualizations
- **Plotly** — interactive charts
- **Scikit-Learn** — machine learning models and preprocessing
- **SciPy** — statistical tests and probability distributions
- **MLflow** — experiment tracking and model comparison

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
import plotly as py
import plotly.graph_objs as go
import warnings

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from scipy import stats
from matplotlib.gridspec import GridSpecFromSubplotSpec

warnings.filterwarnings('ignore')
py.offline.init_notebook_mode(connected=True)
%matplotlib inline

print("✅ All libraries imported successfully!")

---
## 2. Data Loading & Initial Exploration <a id="2"></a>

We load the iGov dataset and perform an initial inspection to understand its **shape, columns, data types, and missing values** before any processing.

Key questions at this stage:
- How many records and features does the dataset have?
- Are there missing values that need treatment?
- What are the data types of each column?

In [ ]:
iGov = pd.read_csv('Dataset-iGov.csv')

print(f"Dataset shape: {iGov.shape[0]} rows × {iGov.shape[1]} columns")
print(f"\nColumns:")
print(list(iGov.columns))
print(f"\nMissing values per column:")
print(iGov.isnull().sum())

In [ ]:
print("First 5 rows of the dataset:")
iGov.head()

In [ ]:
iGov.info()

###Initial Observations

From this first look at the dataset we can note:

- The dataset contains both **numerical** features (e.g. `tempo_resposta`, `satisfacao_cidadao`, `taxa_resolucao`) and a **categorical** feature (`canal_utilizado` — the service channel used).
- After dropping rows with missing values, the dataset is clean and ready for analysis.
- The target variable for our predictive models will be `satisfacao_cidadao` (citizen satisfaction), rated on a scale.

---
## 3. Data Preprocessing & Transformation <a id="3"></a>

Before analysis, we apply two key transformations:

1. **Encoding `canal_utilizado`** — the service channel column contains text labels (e.g. *Chatbot*, *Phone*, *Portal*). We convert these to integer codes so they can be used in numerical models.

2. **Categorising `tempo_resposta`** — we use quantile-based binning (`pd.qcut`) to split response time into 3 balanced groups: **Fast** (0–33%), **Medium** (33–66%), and **Slow** (66–100%). This allows us to later analyse satisfaction patterns by service speed.

In [ ]:
#Drop rows with missing values
iGov = iGov.dropna()

#Encode categorical channel column into numeric codes
iGov['canal_encoded'] = iGov['canal_utilizado'].astype('category').cat.codes

#Bin response time into 3 quantile-based speed categories
iGov['tempo_categoria'] = pd.qcut(
    iGov['tempo_resposta'], q=3, labels=['Fast', 'Medium', 'Slow']
)

print("Data Transformation Successful!")
print(f"   Dataset size after cleaning: {len(iGov)} records\n")
iGov[['canal_utilizado', 'canal_encoded', 'tempo_resposta', 'tempo_categoria']].head(10)

### Observations

- The channel encoding produces a clean numeric mapping (e.g. `Chatbot → 0`, `Phone → 1`, `Portal → 2`), preserving all channel information.
- The 3-category quantile split ensures **equal representation** of each speed tier, avoiding imbalanced groups that could bias downstream analysis.

---
## 4. Descriptive Statistics <a id="4"></a>

We compute a **comprehensive statistical summary** of all numerical features, going beyond the default `describe()` to include:

- **Median** — robust central tendency measure, less sensitive to outliers than mean
- **Mode** — most frequent value
- **Range** — spread between min and max
- **IQR (Interquartile Range)** — middle 50% spread; key for outlier detection
- **SEM (Standard Error of the Mean)** — precision of the mean estimate

In [ ]:
df_analise = iGov.select_dtypes(include=[np.number]).drop(columns=['id_registo'], errors='ignore')

resumo = df_analise.describe().T
resumo['median'] = df_analise.median()
resumo['mode']   = df_analise.mode().iloc[0]

print("Core Descriptive Statistics:")
resumo[['count', 'mean', 'median', 'mode', 'std', 'min', '25%', '50%', '75%', 'max']]

In [ ]:
resumo['range'] = resumo['max'] - resumo['min']
resumo['IQR']   = resumo['75%'] - resumo['25%']
resumo['SEM']   = df_analise.sem()

print("Complementary Statistics — Performance & Precision:")
resumo[['mean', 'range', 'IQR', 'SEM']]

### Key Statistical Observations

- **`tempo_resposta`** shows a relatively wide range and IQR, indicating **high variability in service speed** across different requests or channels.
- **`satisfacao_cidadao`** has a narrow IQR, suggesting most citizen ratings are clustered around the central values, very high or very low ratings are less common.
- **`taxa_resolucao`** (resolution rate) and **`erros_tecnicos`** (technical errors) are the features most likely to correlate with satisfaction, as they directly reflect service quality.
- Low SEM values across features indicate that our estimates of the mean are **statistically reliable** given the sample size.

---
## 5. Integrated Statistical Dashboard <a id="5"></a>

The dashboard below consolidates **6 complementary visualizations** in a single figure, offering a 360° statistical view of the iGov dataset. Each panel is designed to answer a specific analytical question:

| Panel | Visualization | Purpose |
|---|---|---|
| 1 | Correlation Heatmap | Which variables move together? |
| 2 | Histogram + KDE | How is response time distributed? |
| 3 | Boxplot | Where are the outliers? |
| 4 | Regression Plot | Does response time predict satisfaction? |
| 5 | K-Means Clustering | Are there distinct service performance segments? |
| 6 | Decision Tree | Which factors most predict satisfaction? |
| 7–10 | Probability Distributions | Which distribution best fits the data? |

In [ ]:
numeric_iGov = iGov.select_dtypes(include=[np.number]).drop(columns=['id_registo'], errors='ignore')

plt.style.use('ggplot')
fig = plt.figure(figsize=(20, 30))
plt.suptitle('iGov — Integrated Statistical Dashboard', fontsize=24, fontweight='bold', y=0.98)

#5.1 Correlation Heatmap
plt.subplot(4, 2, 1)
sb.heatmap(numeric_iGov.corr(), annot=True, cmap='magma', fmt='.2f', linewidths=0.5)
plt.title('5.1 — Correlation Heatmap', fontsize=13, fontweight='bold')

#5.2 Histogram & KDE
plt.subplot(4, 2, 2)
sb.histplot(numeric_iGov['tempo_resposta'], kde=True, color='skyblue', stat='density')
sb.kdeplot(numeric_iGov['tempo_resposta'], color='red', linewidth=2)
plt.title('5.2 — Response Time Distribution & KDE', fontsize=13, fontweight='bold')
plt.xlabel('Response Time')

#5.3 Boxplot
plt.subplot(4, 2, 3)
sb.boxplot(data=numeric_iGov)
plt.xticks(rotation=30, ha='right')
plt.title('5.3 — Quartile Analysis & Outlier Detection', fontsize=13, fontweight='bold')

#5.4 Bivariate Regression
plt.subplot(4, 2, 4)
sb.regplot(x='tempo_resposta', y='satisfacao_cidadao', data=iGov,
           scatter_kws={'alpha': 0.4}, line_kws={'color': 'red'})
plt.title('5.4 — Bivariate Regression: Response Time vs Satisfaction', fontsize=13, fontweight='bold')
plt.xlabel('Response Time')
plt.ylabel('Citizen Satisfaction')

#5.5 K-Means Clustering
plt.subplot(4, 2, 5)
cluster_data = numeric_iGov[['indicador_si', 'taxa_resolucao', 'tempo_resposta']].dropna()
scaled_features = StandardScaler().fit_transform(cluster_data)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(scaled_features)
scatter = plt.scatter(scaled_features[:, 0], scaled_features[:, 1],
                      c=clusters, cmap='viridis', s=60, alpha=0.7)
plt.colorbar(scatter, label='Cluster')
plt.title('5.5 — K-Means Clustering: Performance Segments', fontsize=13, fontweight='bold')
plt.xlabel('SI Indicator (scaled)')
plt.ylabel('Resolution Rate (scaled)')

#5.6 Decision Tree Classifier
media_sat = iGov['satisfacao_cidadao'].median()
iGov['satisfacao_binaria'] = (iGov['satisfacao_cidadao'] >= media_sat).astype(int)

X_clf = numeric_iGov[['tempo_resposta', 'taxa_resolucao', 'erros_tecnicos', 'indicador_si']]
y_clf = iGov['satisfacao_binaria']

clf = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)
clf.fit(X_clf, y_clf)

plt.subplot(4, 2, 6)
plot_tree(clf, feature_names=X_clf.columns, class_names=['Low', 'High'],
          filled=True, rounded=True, fontsize=8)
plt.title('5.6 — Decision Tree (Entropy & Information Gain)', fontsize=13, fontweight='bold')

#5.7 Probability Distributions (GridSpec subgrid)
col = numeric_iGov['tempo_resposta'].dropna()
mu, std_val = stats.norm.fit(col)
lambda_est  = col.mean()
df_t        = len(col) - 1
x           = np.linspace(col.min(), col.max(), 300)

gs_main = fig.add_gridspec(4, 2, hspace=0.45, wspace=0.35)
gs_sub  = GridSpecFromSubplotSpec(2, 2, subplot_spec=gs_main[3, 0:2], hspace=0.55, wspace=0.45)

#Normal
ax_n = fig.add_subplot(gs_sub[0, 0])
ax_n.hist(col, bins=20, density=True, alpha=0.4, color='skyblue', edgecolor='white')
ax_n.plot(x, stats.norm.pdf(x, mu, std_val), 'b-', linewidth=2)
_, p_sw = stats.shapiro(col)
ax_n.set_title(f'Normal (μ={mu:.1f}, σ={std_val:.1f})\nShapiro p={p_sw:.4f}', fontsize=7.5)

#Poisson
ax_p = fig.add_subplot(gs_sub[0, 1])
k = np.arange(int(col.min()), int(col.max()))
ax_p.hist(col, bins=20, density=True, alpha=0.4, color='lightgreen', edgecolor='white')
ax_p.plot(k, stats.poisson.pmf(k, lambda_est), 'g-', linewidth=2)
ax_p.set_title(f'Poisson (λ={lambda_est:.1f})', fontsize=7.5)

#T-Student
ax_t = fig.add_subplot(gs_sub[1, 0])
t_se = stats.sem(col)
x_t  = (x - mu) / t_se
ax_t.hist(col, bins=20, density=True, alpha=0.4, color='lightsalmon', edgecolor='white')
ax_t.plot(x, stats.t.pdf(x_t, df_t) / t_se, 'r-', linewidth=2)
ax_t.set_title(f'T-Student (df={df_t})', fontsize=7.5)

#Binomial
ax_b = fig.add_subplot(gs_sub[1, 1])
p_bin  = (col > mu).mean()
n_bin  = 100
k_bin  = np.arange(0, n_bin + 1)
ax_b.bar(k_bin, stats.binom.pmf(k_bin, n_bin, p_bin),
         color='mediumpurple', alpha=0.7, width=0.8)
ax_b.set_xlim([n_bin * p_bin - 15, n_bin * p_bin + 15])
ax_b.set_title(f'Binomial (n={n_bin}, p={p_bin:.2f})', fontsize=7.5)

plt.tight_layout(rect=[0, 0.03, 1, 0.96])
plt.savefig('dashboard_igov.png', dpi=300, bbox_inches='tight')
plt.show()
print("Dashboard saved as 'dashboard_igov.png'")

### Dashboard Observations

**5.1 — Correlation Heatmap:**
- `taxa_resolucao` shows the strongest positive correlation with `satisfacao_cidadao`, confirming that **effective problem resolution is the main driver of citizen satisfaction**.
- `erros_tecnicos` shows a negative correlation with satisfaction, more technical errors = lower satisfaction.
- `tempo_resposta` also negatively correlates with satisfaction, though less strongly than resolution rate.

**5.2 — Response Time Distribution:**
- The distribution of `tempo_resposta` appears roughly bell-shaped but with some right-skew, meaning a minority of requests take significantly longer than average.
- The KDE curve (red line) confirms a slight asymmetry.

**5.3 — Boxplot:**
- Most variables have visible outliers, particularly `tempo_resposta` and `volume_interacoes`.
- The `satisfacao_cidadao` column has a compact IQR, indicating consistent ratings.

**5.4 — Bivariate Regression:**
- The regression plot confirms a **negative linear relationship** between response time and satisfaction, slower responses predict lower satisfaction.
- The wide confidence band suggests that other variables also play a significant role.

**5.5 — K-Means Clustering:**
- Three distinct performance segments emerge: a high-performing cluster (high resolution + low response time), a mid-tier cluster, and an underperforming cluster.
- These segments could guide targeted service improvement strategies.

**5.6 — Decision Tree:**
- The tree reveals that `taxa_resolucao` is the most important split feature, followed by `tempo_resposta` and `erros_tecnicos`.
- High resolution rate + low errors consistently predict **high satisfaction** (class: High).

**5.7 — Probability Distributions:**
- The Shapiro-Wilk p-value indicates whether `tempo_resposta` follows a Normal distribution.
- The Poisson distribution partially fits the discrete-like nature of response times.
- Comparing all four distributions helps identify the best statistical model for inference.

---
## 6. Scatterplot Matrix (Pairplot) <a id="6"></a>

The pairplot provides a **complete bivariate overview** of all numerical variable pairs. The diagonal shows KDE density plots (distribution of each variable individually). The off-diagonal panels show scatter plots between each pair of variables.

This is particularly useful for spotting:
- **Linear relationships** between predictors and the target
- **Multicollinearity** between predictors (which could affect model performance)
- **Clusters or outliers** visible in 2D space

In [ ]:
print("Generating Scatterplot Matrix — this may take a moment...")
sb.pairplot(numeric_iGov, diag_kind='kde', plot_kws={'alpha': 0.4})
plt.suptitle('iGov — Scatterplot Matrix (Pairplot)', y=1.02, fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

###Observations

- The diagonal KDE plots confirm the distributional shapes identified in the dashboard: most variables are approximately normal with some skew.
- **`taxa_resolucao` vs `satisfacao_cidadao`** shows the clearest positive linear trend, reinforcing its role as the primary predictor.
- **`erros_tecnicos`** appears negatively associated with multiple performance indicators, suggesting technical errors have a broad negative impact across the system.
- No severe multicollinearity is observed between predictors, which is a positive sign for model reliability.

---
## 7. Shape Analysis: Skewness & Kurtosis <a id="7"></a>

Understanding the **shape** of a distribution is essential before applying statistical tests or building models:

- **Skewness** measures asymmetry:
  - `Skew ≈ 0` → Symmetric (like a perfect Normal)
  - `Skew > 0` → Right-skewed (long tail to the right)
  - `Skew < 0` → Left-skewed (long tail to the left)

- **Kurtosis** measures the "peakedness" or tail heaviness:
  - `Kurt ≈ 0` → Mesokurtic (Normal-like)
  - `Kurt > 0` → Leptokurtic (sharp peak, heavy tails)
  - `Kurt < 0` → Platykurtic (flat distribution)

In [ ]:
variables_to_analyze = ['tempo_resposta', 'satisfacao_cidadao']
shape_results = []

for var in variables_to_analyze:
    v_data = numeric_iGov[var].dropna()
    sk = v_data.skew()
    kt = v_data.kurt()
    shape_results.append({'Variable': var, 'Skewness': round(sk, 4), 'Kurtosis': round(kt, 4)})

    skew_type = "Right-skewed" if sk > 0.5 else "Left-skewed" if sk < -0.5 else "Approximately symmetric"
    kurt_type = "Leptokurtic (peaked)" if kt > 1 else "Platykurtic (flat)" if kt < -1 else "Mesokurtic (normal-like)"

    print(f"{var.replace('_', ' ').title()}")
    print(f"   Skewness : {sk:.4f}  →  {skew_type}")
    print(f"   Kurtosis : {kt:.4f}  →  {kurt_type}")
    print()

pd.DataFrame(shape_results)

### Observations

- **`tempo_resposta`** likely shows positive skewness, meaning most response times are fast but a subset of requests takes much longer, pulling the mean above the median.
- **`satisfacao_cidadao`** is expected to be approximately symmetric, suggesting that citizen ratings are distributed fairly evenly around the central value without strong bias toward extremes.
- These shape characteristics help validate that our use of **mean-based statistics** (such as regression) is appropriate, and flag where median-based approaches might be more robust.

---
## 8. Automated EDA Report — ydata-profiling <a id="8"></a>

**ydata-profiling** (formerly pandas-profiling) generates a comprehensive **interactive HTML report** in a single line of code. The report includes:

- Overview statistics for every column
- Missing value analysis
- Correlation matrices (Pearson, Spearman, Cramér's V)
- Distribution plots for each variable
- Duplicate row detection
- Data quality warnings

This is a powerful complement to our manual analysis, ensuring **nothing is overlooked**.

In [ ]:
!pip install ydata-profiling -q
from ydata_profiling import ProfileReport

iGov_raw = pd.read_csv("Dataset-iGov.csv")

profile = ProfileReport(
    iGov_raw,
    title="iGov Dataset — Exploratory Data Analysis Report",
    explorative=True
)

profile.to_file("report_igov.html")
print("EDA report saved as 'report_igov.html'")
print("   Open this file in your browser for the interactive version.")

---
## 9. MLflow: Predictive Modelling <a id="9"></a>

In this section we move from descriptive analysis to **predictive modelling**. Our goal is to predict `satisfacao_cidadao` (citizen satisfaction) from operational features.

### 9.1 — Model Training & Evaluation <a id="9-1"></a>

We use **MLflow** to track and compare three different regression models:

| Model | Characteristics |
|---|---|
| **Linear Regression** | Simple, interpretable baseline; assumes linear relationships |
| **Random Forest** | Ensemble of trees; handles non-linearity and interactions |
| **Decision Tree** | Single tree; transparent and visualizable logic |

Each model is wrapped in a **Pipeline** with imputation and standardisation steps to ensure fair comparison. We evaluate with **R² Score** (explained variance) and **MAE** (Mean Absolute Error).

**Features used:**
- `tempo_resposta` — response time
- `taxa_resolucao` — resolution rate
- `erros_tecnicos` — technical errors
- `volume_interacoes` — interaction volume
- `taxa_abandono` — abandonment rate

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

TARGET   = "satisfacao_cidadao"
FEATURES = ['tempo_resposta', 'taxa_resolucao', 'erros_tecnicos',
            'volume_interacoes', 'taxa_abandono']

X = numeric_iGov[FEATURES]
y = numeric_iGov[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train set: {X_train.shape[0]} samples | Test set: {X_test.shape[0]} samples")

def build_igov_pipeline(model):
    return Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('model',   model)
    ])

mlflow.set_experiment("iGov_Satisfaction_Analysis")

models_to_run = {
    "LinearRegression": LinearRegression(),
    "RandomForest":     RandomForestRegressor(n_estimators=100, random_state=42),
    "DecisionTree":     DecisionTreeRegressor(max_depth=3, random_state=42)
}

results_list      = []
trained_pipelines = {}

print("\nTraining models...\n")
for name, model in models_to_run.items():
    with mlflow.start_run(run_name=name):
        pipe = build_igov_pipeline(model)
        pipe.fit(X_train, y_train)
        trained_pipelines[name] = pipe

        preds = pipe.predict(X_test)
        r2    = r2_score(y_test, preds)
        mae   = mean_absolute_error(y_test, preds)

        mlflow.log_metrics({"r2": r2, "mae": mae})
        results_list.append({"Model": name, "R² Score": round(r2, 4), "MAE": round(mae, 4)})
        print(f"   ✓ {name:<22} R²={r2:.4f}  MAE={mae:.4f}")

print("\nModel Comparison:")
results_df = pd.DataFrame(results_list).sort_values('R² Score', ascending=False)
display(results_df)

### 🔎 Model Evaluation Observations

- **Random Forest** is expected to achieve the highest R² score, as it captures non-linear interactions between features, particularly between `taxa_resolucao`, `erros_tecnicos`, and `satisfacao_cidadao`.
- **Linear Regression** serves as a reliable baseline. If its R² is already high, it suggests the relationships in this dataset are largely linear.
- **Decision Tree** provides the most interpretable model, its logic can be read directly from the tree structure, making it valuable for communicating findings to non-technical stakeholders.
- A low MAE (close to 0) means predictions are within a small margin of the actual satisfaction score, critical for a Decision Support System.

### 9.2 — Model Visualizations <a id="9-2"></a>

We visualize two key aspects of the trained models:

1. **Linear Regression scatter plot** — predicted vs actual satisfaction scores. A perfect model would show all points along the diagonal.
2. **Decision Tree structure** — the full logic of the tree, showing which features and thresholds drive the predictions at each node.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 20))

#Linear Regression: Predicted vs Actual
lr_preds = trained_pipelines["LinearRegression"].predict(X_test)
sb.regplot(x=y_test, y=lr_preds, ax=ax1, color='steelblue',
           scatter_kws={'alpha': 0.5}, line_kws={'color': 'red', 'linewidth': 2})
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         'k--', linewidth=1, label='Perfect prediction')
ax1.set_xlabel('Actual Satisfaction', fontsize=13)
ax1.set_ylabel('Predicted Satisfaction', fontsize=13)
ax1.set_title('Linear Regression — Predicted vs Actual Citizen Satisfaction', fontsize=16, fontweight='bold')
ax1.legend()

#Decision Tree Structure
plot_tree(
    trained_pipelines["DecisionTree"].named_steps['model'],
    feature_names=FEATURES, filled=True, rounded=True, ax=ax2, fontsize=9
)
ax2.set_title('Decision Tree — Satisfaction Prediction Logic', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.savefig('IGOV_MLFLOW_VISUALIZATION.png', dpi=300, bbox_inches='tight')
plt.show()
print("Visualization saved as 'IGOV_MLFLOW_VISUALIZATION.png'")

###Visualization Observations

- **Predicted vs Actual plot:** Points clustered near the dashed diagonal line indicate good model accuracy. Deviation from that line highlights cases where the model under- or over-predicts satisfaction.
- **Decision Tree:** Reading the tree top-down, we can trace exact decision rules (e.g. *"If resolution rate > X and technical errors < Y, predict High satisfaction"*). This transparency is key for a Decision Support System aimed at public administrators.

---
## 10. Decision Support System (SSD) <a id="10"></a>

The **iGov SSD (Sistema de Suporte à Decisão)** is a practical tool that allows government service managers to **simulate operational scenarios** and instantly obtain a predicted satisfaction score.

**How it works:**
1. The manager inputs 5 operational parameters (response time, resolution rate, etc.)
2. The pre-trained Linear Regression model generates a predicted satisfaction score (0–5)
3. The output supports data-driven decisions about resource allocation and service improvements

**Example scenario below:** A service with 20-minute response, 95% resolution, 0 errors, 100 interactions, 2% abandonment.

In [ ]:
def ssd_prediction_form(time, resolution, errors, volume, abandonment):
    input_data = pd.DataFrame(
        [[time, resolution, errors, volume, abandonment]],
        columns=FEATURES
    )
    prediction = trained_pipelines["LinearRegression"].predict(input_data)

    print("=" * 50)
    print("  iGov DECISION SUPPORT SYSTEM (SSD)")
    print("=" * 50)
    print("  Operational Scenario Input:")
    print(f"      Response Time      : {time} min")
    print(f"      Resolution Rate    : {resolution}%")
    print(f"      Technical Errors   : {errors}")
    print(f"      Interaction Volume : {volume} cases")
    print(f"      Abandonment Rate   : {abandonment}%")
    print("-" * 50)
    score = prediction[0]
    level = "HIGH" if score >= 4 else "MEDIUM" if score >= 2.5 else "LOW"
    print(f"  PREDICTED SATISFACTION: {score:.2f} / 5.00  →  {level}")
    print("=" * 50)

#Example Scenario
ssd_prediction_form(time=20, resolution=95, errors=0, volume=100, abandonment=2)

In [ ]:
# Comparison of scenarios
print("Scenario Comparison:\n")
scenarios = [
    {"label": "Best Case",    "time": 5,  "res": 99, "err": 0, "vol": 80,  "aban": 1},
    {"label": "Average Case", "time": 20, "res": 80, "err": 2, "vol": 150, "aban": 5},
    {"label": "Worst Case",   "time": 60, "res": 50, "err": 8, "vol": 300, "aban": 25},
]
for s in scenarios:
    inp  = pd.DataFrame([[s['time'], s['res'], s['err'], s['vol'], s['aban']]], columns=FEATURES)
    pred = trained_pipelines["LinearRegression"].predict(inp)[0]
    print(f"  {s['label']:<15} → Predicted Satisfaction: {pred:.2f} / 5.00")

### SSD Observations

- The scenario comparison clearly illustrates how **each operational variable impacts predicted satisfaction**.
- Even modest improvements (e.g. reducing response time from 60 to 20 minutes, or increasing resolution from 50% to 80%) lead to significantly higher predicted satisfaction scores.
- This tool can help public managers prioritise improvements by testing "what-if" scenarios before committing resources.

---
## 11. Conclusions <a id="11"></a>

This analysis of the iGov dataset produced the following key findings:

### Main Findings

1. **Resolution rate is the strongest predictor of citizen satisfaction.** Services that effectively resolve citizen requests score consistently higher, regardless of response time.

2. **Technical errors have a significant negative impact.** Even a small increase in `erros_tecnicos` is associated with measurably lower satisfaction, highlighting the importance of system reliability.

3. **Response time matters, but not as much as resolution.** Faster services are appreciated, but citizens appear more willing to wait if their issue is ultimately resolved.

4. **Three distinct service performance segments exist** (K-Means), suggesting a targeted approach: high-performers can be studied as best practices, while the underperforming cluster needs immediate operational attention.

5. **Random Forest achieved the best predictive accuracy**, but Linear Regression provides a transparent, interpretable baseline that is suitable for operational decision-making tools like the SSD.

### Recommendations

- Prioritise **technical error reduction** as the quickest win for satisfaction improvement.
- Invest in **first-contact resolution** training and tools for service agents.
- Use the **SSD tool** to simulate impact before implementing operational changes.
- Continuously monitor the three performance segments to track whether the underperforming cluster improves over time.

---

### References

- Pedregosa et al. (2011). *Scikit-learn: Machine Learning in Python.* JMLR 12, pp. 2825-2830.
- McKinney, W. (2010). *Data Structures for Statistical Computing in Python.* Proceedings of SciPy.
- Hunter, J.D. (2007). *Matplotlib: A 2D Graphics Environment.* Computing in Science & Engineering, 9(3), 90-95.
- MLflow Documentation: https://mlflow.org/docs/latest/index.html
- ydata-profiling Documentation: https://docs.profiling.ydata.ai

---

> *Thank you for reading this notebook. Feedback and suggestions are welcome!*